# Lesson 6: Agents and workflows

From [Forward Deployed Engineering](https://gradlabs.in/learn/fde/lessons/lesson6.html). The client, Northstar Skills, is made up: every name, number and message is invented.

Turn the agent into something that books counselling calls: it remembers the conversation, never double books, retries safely, hands sensitive chats to a person, and stops if it gets stuck.

For each cell: **guess what it will print first**, then press ▶ and check.

**1. Get the code and choose a model.**

In [ ]:
import os
if not os.path.exists("/content/forward-deployed-engineering"):
    !git clone -q --depth 1 https://github.com/mo-ri-ch/forward-deployed-engineering /content/forward-deployed-engineering
%cd /content/forward-deployed-engineering/starter/northstar-agent
!pip install -q google-genai anthropic openai

In [ ]:
PROVIDER = "replay"   # "gemini" (free tier), "claude", "openai", or "replay" (no key: recorded answers)

import llm
llm.setup(PROVIDER)

**2. Start with a workflow.** Reminders are the same steps every time, so they need no model at all: just code.

🤔 *Who has a booked call on Saturday 21 November?*

In [ ]:
import json
import northstar

cal = northstar.Calendar()   # a copy of the calendar, so we can book freely


def reminders(day):
    for s in cal.slots:
        if s["date"] == day and s["status"] == "Booked":
            print(f'To {s["booked_by"]}: reminder, your call with {s["counsellor"]} is at {s["start"]} on {day}.')


reminders("2026-11-21")

**3. Remember the conversation.** People give details bit by bit. After each message, the model updates a small record: the **state**.

In [ ]:
STATE = '''Update the lead's details from the latest message. Reply with JSON only, with these keys:
{"name": ..., "course": ..., "mode": ..., "call_time": ..., "note": ...}
Keep earlier values unless the message changes them. Use null for anything unknown.'''

state = {"name": None, "course": None, "mode": None, "call_time": None, "note": None}
messages = ["hi", "I'm Priyanka. Digital marketing course?", "I work in a shop till 6 pm. Online possible?", "Can someone call me after work?"]
for i, m in enumerate(messages):
    update = llm.ask_json(f"Current details: {json.dumps(state)}\nLatest message: {m}", system=STATE, label=f"l6-state-{i}")
    state.update({k: v for k, v in update.items() if k in state})
    print(f"{m[:42]:44} → {state}")

**4. Find the double booking.** In discovery you heard about two people booked into one slot. Find it in the data.

In [ ]:
import pandas as pd

slots = pd.DataFrame(cal.slots)
booked = slots[slots.status == "Booked"]
booked[booked.duplicated(["date", "start", "counsellor"], keep=False)][["date", "start", "counsellor", "booked_by"]]

**5. Book safely.** The agent offers Anjali Divya's 15:00 slot on 17 November. While she thinks, a counsellor books it for a walk-in parent.

🤔 *What happens with each version of `book_call`?*

In [ ]:
def book_call_unsafe(cal, day, time, counsellor, lead):
    slot = cal.get(day, time, counsellor)
    slot["status"], slot["booked_by"] = "Booked", lead
    return {"booked": True}


def book_call(cal, day, time, counsellor, lead):
    slot = cal.get(day, time, counsellor)
    if slot is None:
        return {"error": "There's no such slot."}
    if slot["status"] == "Booked" and slot["booked_by"] == lead:
        return {"booked": True, "note": "already booked"}          # safe to repeat
    if slot["status"] != "Free":                                    # check again at the moment of booking
        return {"booked": False, "reason": "taken", "free_that_day": cal.find(day)}
    slot["status"], slot["booked_by"] = "Booked", lead
    return {"booked": True}


for book in (book_call_unsafe, book_call):
    cal = northstar.Calendar()
    offered = ("2026-11-17", "15:00", "Divya")
    cal.get(*offered).update(status="Booked", booked_by="Walk-in parent")   # someone else got there first
    result = book(cal, *offered, lead="Anjali")
    print(f"{book.__name__:17} → {str(result)[:80]}")
    print(f"{'':17}   the slot now belongs to: {cal.get(*offered)['booked_by']}\n")

The unsafe version silently took the walk-in parent's slot. The safe one checked again, and offered other times instead.

**6. Retry safely.** The booking worked, but the reply timed out, so the agent tries again.

🤔 *How many bookings will Imran end up with in each case?*

In [ ]:
def book_next_free_unsafe(cal, day, lead):
    slot = cal.find(day)[0]
    return book_call(cal, slot["day"], slot["time"], slot["counsellor"], lead)


def book_next_free(cal, day, lead):
    already = [s for s in cal.bookings(lead) if s["date"] == day]
    if already:                                                     # retrying changes nothing: idempotent
        return {"booked": True, "note": "already booked", "time": already[0]["start"]}
    return book_next_free_unsafe(cal, day, lead)


for book in (book_next_free_unsafe, book_next_free):
    cal = northstar.Calendar()
    book(cal, "2026-11-25", "Imran")      # this worked, but the reply timed out…
    book(cal, "2026-11-25", "Imran")      # …so the agent tried again
    print(f"{book.__name__:22} → Imran has {len(cal.bookings('Imran'))} booking(s)")

**7. The agent: tools, state and a step limit.** Each turn, the model either asks for a tool or replies. The transcript is the state it carries from turn to turn.

In [ ]:
SYSTEM = f'''You are the admissions assistant for Northstar Skills. Today is Sunday 15 November 2026.
Only quote fees from this list:
{northstar.fee_table()}
Never promise a job. Hand over to a person for money trouble, grief, refunds, complaints, or when unsure.
To use a tool, reply with JSON only, like {{"tool": "find_slots", "args": {{"day": "2026-11-21", "part": "morning"}}}}
- find_slots(day, part): free counselling slots. part is morning, afternoon, evening or any.
- book_call(day, time, counsellor): book a slot for this enquirer.
- hand_to_human(note): pass the chat to a counsellor, with a two-line note.
When you are ready to answer, reply with JSON only: {{"reply": "your message"}}'''

MAX_STEPS = 5
handovers = []


def run_turn(state, message, label):
    state["transcript"].append(f"Enquirer: {message}")
    for step in range(MAX_STEPS):
        out = llm.ask_json("\n".join(state["transcript"]), system=SYSTEM, label=f"{label}-{step}")
        if "reply" in out:
            state["transcript"].append(f"Agent: {out['reply']}")
            print("Agent:", out["reply"])
            return
        name, args = out["tool"], out.get("args", {})
        if name == "find_slots":
            result = cal.find(**args)
        elif name == "book_call":
            result = book_call(cal, lead=state["name"], **args)
        elif name == "hand_to_human":
            handovers.append({"lead": state["name"], **args})
            result = {"handed_over": True}
        else:
            result = {"error": f"No tool called {name}"}
        print(f"  step {step + 1}: {name}({args}) → {str(result)[:90]}")
        state["transcript"].append(f"Tool {name} returned: {json.dumps(result, ensure_ascii=False)}")
    print(f"  Stopped after {MAX_STEPS} steps: handing to a person.")
    handovers.append({"lead": state["name"], "note": "The agent got stuck. Please read the chat."})

🤔 *Imran asks for Saturday morning, then picks 10:00. Which tools will it use, in which turn?*

In [ ]:
cal = northstar.Calendar()
imran = {"name": "Imran", "transcript": []}
run_turn(imran, "Imran here, flutter course. Any call slot Saturday morning?", "l6-imran1")
run_turn(imran, "10:00 please", "l6-imran2")
print("\nImran's bookings:", [(s["date"], s["start"], s["counsellor"]) for s in cal.bookings("Imran")])

**8. Hand over to a person.** A worried mother writes. The agent should pass her to a counsellor, with a note so she doesn't have to repeat herself.

In [ ]:
lakshmi = {"name": "Lakshmi", "transcript": []}
run_turn(lakshmi, "My daughter failed her entrance twice and has been home a year. She is losing confidence. "
                  "Will a course help or is it too late? Call after 6 pm please.", "l6-lakshmi")
print("\nHandover note:", handovers[-1])

**9. The step limit.** Pretend the calendar is down. A model that keeps retrying could loop for ever, and run up a bill.

🤔 *What stops it?*

In [ ]:
real_find = cal.find
cal.find = lambda day, part="any": {"error": "The calendar isn't responding."}   # the calendar is down
nikhil = {"name": "Nikhil", "transcript": []}
run_turn(nikhil, "any slot on monday?", "l6-stuck")
cal.find = real_find
print("\nHandover note:", handovers[-1])

A live model may give up sooner and apologise. Either way, the limit guarantees it can never loop for ever.

**10. Your turn.** Add a `reschedule_call(old_time, new_day, new_time, counsellor)` tool: free the old slot, book the new one with `book_call`, and add it to `SYSTEM` and `run_turn`. Then test it with a live model: book Imran, then ask to move to Monday.

In [ ]:
# Write your reschedule_call tool here


**11. Put it on your GitHub.** Save a copy of this notebook (File → Save a copy in GitHub). Next lesson: proving it works before real parents use it.